In [ ]:
import pandas as pd
from pygments import lex
from pygments.lexers import get_lexer_by_name, guess_lexer
from pygments.util import ClassNotFound
from pygments.token import Name, Literal

# =========================
# CONFIG
# =========================
INPUT_CSV = "data/filtered_H-AIRosettaMP.csv"
OUTPUT_CSV = "output_tokenized.csv"

CODE_COLUMN = "code"
LANG_COLUMN = None  # 👈 importante: NO string

LANG_MAP = {
    "python": "python",
    "java": "java",
    "c++": "cpp",
    "cpp": "cpp",
    "c#": "csharp",
    "csharp": "csharp",
    "js": "javascript",
    "javascript": "javascript"
}

# =========================
# FUNCIONES
# =========================
def get_lexer(code, lang=None):
    if lang:
        lang = lang.lower()
        if lang in LANG_MAP:
            try:
                return get_lexer_by_name(LANG_MAP[lang])
            except ClassNotFound:
                pass

    try:
        return guess_lexer(code)
    except ClassNotFound:
        return None


def normalize_token(tok_type, tok_value):
    if tok_type in Name:
        return "VAR"
    elif tok_type in Literal:
        return "CONST"
    else:
        return tok_value


def tokenize_code(code, lang=None):
    lexer = get_lexer(code, lang)
    if lexer is None:
        return ""

    tokens = []
    for tok_type, tok_value in lex(code, lexer):
        tok_value = tok_value.strip()
        if tok_value:
            tokens.append(normalize_token(tok_type, tok_value))

    return " ".join(tokens)

# =========================
# PIPELINE
# =========================
def main():
    df = pd.read_csv(INPUT_CSV)

    if CODE_COLUMN not in df.columns:
        raise ValueError(f"No existe la columna '{CODE_COLUMN}'")

    tokenized = []

    for i, row in df.iterrows():
        code = str(row[CODE_COLUMN])
        lang = str(row[LANG_COLUMN]) if LANG_COLUMN else None

        tokens = tokenize_code(code, lang)
        tokenized.append(tokens)

        if i % 1000 == 0:
            print(f"Procesados: {i}")

    
    df["tokens"] = tokenized
    df.to_csv(OUTPUT_CSV, index=False)

    print("✅ Tokenización completada")

if __name__ == "__main__":
    main()

Procesados: 0
Procesados: 1000
Procesados: 2000
Procesados: 3000
Procesados: 4000
Procesados: 5000
Procesados: 6000
Procesados: 7000
Procesados: 8000
Procesados: 9000
Procesados: 10000
✅ Tokenización completada
